# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. 

Source: Original local package: 01_google_mlcc/05b_fairness_portable.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '01_google_mlcc', 'torch', '05b_fairness_portable.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

# Portable fairness exercise: group metrics and MMD remediation
Source exercise: https://github.com/google/eng-edu/blob/main/ml/cc/exercises/fairness_income.ipynb
This **original PyTorch adaptation** implements the same positive-subset idea and a Gaussian-kernel MMD penalty. It is not a run of TensorFlow Model Remediation or the TFMA widget. It uses a deterministic random sample of the actual ACS 2018 course CSV, with all preprocessing fitted on training data. Samples are used only for this educational exercise.


In [ ]:
import copy,json,os
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score,confusion_matrix,accuracy_score
torch.manual_seed(42);torch.set_num_threads(1);np.random.seed(42)
results=Path('results');results.mkdir(exist_ok=True)
raw=pd.read_csv('data/acsincome_raw_2018.csv')
sample=raw.sample(n=min(12000,len(raw)),random_state=42).copy()
sample['PINCP']=(sample.PINCP>50000).astype(int)
idx=np.arange(len(sample));strata=sample.SEX.astype(str)+'_'+sample.PINCP.astype(str)
train,other=train_test_split(idx,test_size=.4,random_state=42,stratify=strata)
valid,test=train_test_split(other,test_size=.5,random_state=43,stratify=strata.iloc[other])
columns=sample.columns.drop('PINCP').tolist()
preprocess=make_pipeline(SimpleImputer(strategy='median'),StandardScaler())
preprocess.fit(sample.iloc[train][columns])
features=torch.tensor(preprocess.transform(sample[columns]),dtype=torch.float32)
labels=torch.tensor(sample.PINCP.to_numpy(),dtype=torch.float32)
groups=sample.SEX.to_numpy()
positive_female=train[(groups[train]==2)&(sample.PINCP.to_numpy()[train]==1)]
positive_male=train[(groups[train]==1)&(sample.PINCP.to_numpy()[train]==1)]
assert len(positive_female)>0 and len(positive_male)>0
assert set(train).isdisjoint(set(valid)|set(test))
print('Positive training subset sizes:',len(positive_female),len(positive_male))

In [ ]:
def new_model():
    return nn.Sequential(nn.Linear(features.shape[1],64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))
def mmd(a,b,bandwidth=.2):
    a=a.reshape(-1,1);b=b.reshape(-1,1)
    kernel=lambda x,y:torch.exp(-torch.cdist(x,y).square()/(2*bandwidth**2))
    return kernel(a,a).mean()+kernel(b,b).mean()-2*kernel(a,b).mean()
assert abs(float(mmd(torch.tensor([.2,.8]),torch.tensor([.2,.8]))))<1e-6
def fit(model,epochs,penalty_weight,seed):
    generator=torch.Generator().manual_seed(seed)
    optimizer=torch.optim.Adam(model.parameters(),lr=.003);history=[]
    for epoch in range(epochs):
        order=torch.as_tensor(train)[torch.randperm(len(train),generator=generator)]
        losses=[]
        for batch in order.split(256):
            model.train();optimizer.zero_grad(set_to_none=True)
            loss=nn.functional.binary_cross_entropy_with_logits(model(features[batch]).ravel(),labels[batch])
            if penalty_weight:
                a=torch.as_tensor(positive_female)[torch.randint(len(positive_female),(64,),generator=generator)]
                b=torch.as_tensor(positive_male)[torch.randint(len(positive_male),(64,),generator=generator)]
                loss=loss+penalty_weight*mmd(model(features[a]).sigmoid(),model(features[b]).sigmoid())
            loss.backward();optimizer.step();losses.append(float(loss.detach()))
        history.append({'epoch':epoch,'objective':float(np.mean(losses))})
    return history
initial=new_model();fit(initial,8,0,42)
control=copy.deepcopy(initial);remediated=copy.deepcopy(initial)
# Match the extra number of optimisation epochs for a fair control comparison.
history_control=fit(control,5,0,123)
history_remediated=fit(remediated,5,1,123)

## Tasks 1 and 4: evaluate AUC and false-negative rate per group


In [ ]:
def group_metrics(model,indices):
    model.eval()
    with torch.no_grad():p=model(features[indices]).sigmoid().ravel().numpy()
    yy=labels[indices].numpy();gg=groups[indices];rows=[]
    for name,mask in [('overall',np.ones(len(indices),dtype=bool)),('female',gg==2),('male',gg==1)]:
        truth=yy[mask];pred=p[mask]>=.5
        tn,fp,fn,tp=confusion_matrix(truth,pred,labels=[0,1]).ravel()
        rows.append({'group':name,'n':int(mask.sum()),'positive_n':int((truth==1).sum()),
          'auc':roc_auc_score(truth,p[mask]) if len(np.unique(truth))==2 else np.nan,
          'accuracy':accuracy_score(truth,pred),'false_negative_rate':fn/(fn+tp) if fn+tp else np.nan,
          'false_positive_rate':fp/(fp+tn) if fp+tn else np.nan})
    return pd.DataFrame(rows)
# We fixed the settings in advance. The test comparison is descriptive, not a tuning loop.
control_metrics=group_metrics(control,test).assign(model='matched control')
remediated_metrics=group_metrics(remediated,test).assign(model='MMD adaptation')
report=pd.concat([control_metrics,remediated_metrics],ignore_index=True)
report.to_csv(results/'acs_portable_fairness_metrics.csv',index=False)
pd.DataFrame(history_control).to_csv(results/'acs_control_training.csv',index=False)
pd.DataFrame(history_remediated).to_csv(results/'acs_remediation_training.csv',index=False)
print(report.to_string(index=False))
for name,frame in report.groupby('model'):
    group_fnr=frame.set_index('group').false_negative_rate
    print(name,'absolute female/male FNR gap:',abs(group_fnr.female-group_fnr.male))
print('Report the observed tradeoff; neither equal AUC nor a lower FNR gap proves fairness.')

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)